# Módulo 4: Construcción de características

## Tratamiento de categóricos I: One hot encoding + ordinal encoding

#### Objetivo 

* Comprender cómo es que los modelos de ML pueden consumir categorías y como éstas pasan a ser números.

#### Actividad 1
Tomar el dataset `actividad_9,1a` y conforme a los tipos de encoders realizar:
* One hot
    * Generar un modelo con todas las variables (incluyendo las categóricas) y otro solo con las numéricas (originales) y comparar el performance.
* Ordinal
    * Repetir paso anterior.

#### Actividad 2
Tomar la base de datos `Automobile_data.csv` de la plataforma y:
* Generar un análisis exploratorio básico.
    * Identifica variables categóricas y numéricas.
    * Identifica si hay valores faltantes.
    * Convierte la variable de respuesta (`price`) a numérica. 
* Investiga la utilería de pandas `get_dummies` y explica para qué sirve y en cuál de los dos métodos vistos en clase puede ser utilizado.
* Investiga la utilería de sklearn `OrdinalEncoder` y explica para qué sirve y en cuál de los dos métodos vistos en clase puede ser utilizado.
* Escoge uno de los dos métodos en clase y encodea tus variables categóricas. 
* Después de generar sets de entrenamiento y de prueba (considera la normalización de los datos en caso de ser necesario) genera un modelo de regresión (`LinearRegression`) con todas tus variables y otro solo con las variables numéricas originales. 
* Mide en los sets de prueba de ambos modelos el performance. 
* Responde:
    * ¿Cuál de los dos métodos (con categóricas / sin categóricas) tuvo mejor performance? ¿Por qué?
    * ¿Qué ventaja tiene usar métodos para encodear variables categóricas? 
    * ¿Qué metodo de encoding utilizarías en un dataset con categorías con alta granularidad? 
    * ¿Qué posible desventaja podría ocurrir en caso de usar un modelo en producción con alguno de los métodos de encodeo vistos en clase?
    
    


<!-- sklearn.datasets.make_regression(n_samples=100, n_features=100, *, n_informative=10, n_targets=1, bias=0.0, effective_rank=None, tail_strength=0.5, noise=0.0, shuffle=True, coef=False, random_state=None -->

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.datasets import make_regression

seed = 28
path = '~.'

Nos hemos encontrado con muchos datasets, la mayoría tenían muchos datos numéricos que pueden ser después consumidos por un modelo.

Sin embargo la vida real es ligeramente (o muy) diferente. 

En la vida real hay categorías inevitables, es decir, cosas que son agrupaciones de características más que datos duros. Algunos ejemplos son:
* Ubicación (sin coordenadas).
* Tipo de producto comprado.
* Estado de nacimiento.
* Mes de aplicación. 

Todos esos son datos que pueden ser convertidos en números siguiendo diversas estrategias.

## One-hot encoding.

Esta metodología nos ayuda a separar en variables sencillas (booleanas) diversas categorías mediante la creación de nuevas columnas. 

In [6]:
df = pd.read_csv('actividad_9,1a.csv')

Tomemos un frame como el siguiente: 

In [7]:
df

,Edad,Estado,Artista_favorito,Salario
0,45.781889,Jalisco,Cristian Castro,31576.333799
1,54.303289,Jalisco,Cristian Castro,47648.067980
2,48.112729,Sinaloa,Luis Miguel,65000.000000
3,65.000000,Jalisco,Luis Miguel,34330.567827
4,27.137411,Jalisco,Luis Miguel,44627.927179
5,18.000000,Jalisco,Luis Miguel,50390.778827
6,24.737619,CDMX,Luis Miguel,15000.000000
7,27.584333,Jalisco,Cristian Castro,23621.972569
8,22.215258,Jalisco,Luis Miguel,37153.991419
9,46.863818,Jalisco,Luis Miguel,41774.881238


El procedimiento de _one hot encoding_ se basa en general $N$ columnas de datos tal que ejemplifique como booleanos las variables provistas. 


Ventajas:
* Es fácil y rápido de replicar.
* Es fácil de leer e interpretar.

Desventajas:
* No sabe tratar con columnas no vistas.
* Variables con alta granularidad generan demasiadas columnas. 

#### Artista favorito

Primer paso: Identificar todas las categorías.

In [8]:
valores_unicos = df['Artista_favorito'].unique()
valores_unicos

<StringArray>
['Cristian Castro', 'Luis Miguel']
Length: 2, dtype: str

Segundo paso: Generar nuevas columnas con todas las categorías.

Tercer paso: Identificar dónde podemos encontrar valores activos. Usaremos `mask`.

In [9]:
df['Artista_favorito_Cristian_Castro'] = df['Artista_favorito'].copy()

In [10]:
#mask 1 y 0
df['Artista_favorito_Cristian_Castro'].mask(df['Artista_favorito_Cristian_Castro'] == 'Cristian Castro', 1)

0              1
1              1
2    Luis Miguel
3    Luis Miguel
4    Luis Miguel
5    Luis Miguel
6    Luis Miguel
7              1
8    Luis Miguel
9    Luis Miguel
Name: Artista_favorito_Cristian_Castro, dtype: object

Ahora solo aplicamos un `inplace`: 

In [11]:
df['Artista_favorito_Cristian_Castro'] = df['Artista_favorito_Cristian_Castro'].mask(df['Artista_favorito_Cristian_Castro'] == 'Cristian Castro', 1)

In [12]:
df['Artista_favorito_Cristian_Castro'] = df['Artista_favorito_Cristian_Castro'].mask(df['Artista_favorito_Cristian_Castro'] == 'Luis Miguel', 0)

Repetimos con la otra categoría.

In [13]:
df

,Edad,Estado,Artista_favorito,Salario,Artista_favorito_Cristian_Castro
0,45.781889,Jalisco,Cristian Castro,31576.333799,1
1,54.303289,Jalisco,Cristian Castro,47648.067980,1
2,48.112729,Sinaloa,Luis Miguel,65000.000000,0
3,65.000000,Jalisco,Luis Miguel,34330.567827,0
4,27.137411,Jalisco,Luis Miguel,44627.927179,0
5,18.000000,Jalisco,Luis Miguel,50390.778827,0
6,24.737619,CDMX,Luis Miguel,15000.000000,0
7,27.584333,Jalisco,Cristian Castro,23621.972569,1
8,22.215258,Jalisco,Luis Miguel,37153.991419,0
9,46.863818,Jalisco,Luis Miguel,41774.881238,0


In [14]:
#lambda
df['Artista_favorito_Luis_Miguel'] = df['Artista_favorito'].apply(
    lambda x: 1 if x == 'Luis Miguel' else 0
)

In [15]:
df

,Edad,Estado,Artista_favorito,Salario,Artista_favorito_Cristian_Castro,Artista_favorito_Luis_Miguel
0,45.781889,Jalisco,Cristian Castro,31576.333799,1,0
1,54.303289,Jalisco,Cristian Castro,47648.067980,1,0
2,48.112729,Sinaloa,Luis Miguel,65000.000000,0,1
3,65.000000,Jalisco,Luis Miguel,34330.567827,0,1
4,27.137411,Jalisco,Luis Miguel,44627.927179,0,1
5,18.000000,Jalisco,Luis Miguel,50390.778827,0,1
6,24.737619,CDMX,Luis Miguel,15000.000000,0,1
7,27.584333,Jalisco,Cristian Castro,23621.972569,1,0
8,22.215258,Jalisco,Luis Miguel,37153.991419,0,1
9,46.863818,Jalisco,Luis Miguel,41774.881238,0,1


Repetimos con la otra categoría, ahora usando otro procedimiento: 

In [17]:
df['Estado'].value_counts()

Estado
Jalisco    8
Sinaloa    1
CDMX       1
Name: count, dtype: int64

In [18]:
datos_unicos = df['Estado'].unique()

In [21]:
for data_unico in datos_unicos:
    df[f'Estado_{data_unico}'] = df['Estado'].apply(lambda x: 1 if x == data_unico else 0)

In [22]:
df

,Edad,Estado,Artista_favorito,Salario,Artista_favorito_Cristian_Castro,Artista_favorito_Luis_Miguel,Estado_Jalisco,Estado_Sinaloa,Estado_CDMX
0,45.781889,Jalisco,Cristian Castro,31576.333799,1,0,1,0,0
1,54.303289,Jalisco,Cristian Castro,47648.067980,1,0,1,0,0
2,48.112729,Sinaloa,Luis Miguel,65000.000000,0,1,0,1,0
3,65.000000,Jalisco,Luis Miguel,34330.567827,0,1,1,0,0
4,27.137411,Jalisco,Luis Miguel,44627.927179,0,1,1,0,0
5,18.000000,Jalisco,Luis Miguel,50390.778827,0,1,1,0,0
6,24.737619,CDMX,Luis Miguel,15000.000000,0,1,0,0,1
7,27.584333,Jalisco,Cristian Castro,23621.972569,1,0,1,0,0
8,22.215258,Jalisco,Luis Miguel,37153.991419,0,1,1,0,0
9,46.863818,Jalisco,Luis Miguel,41774.881238,0,1,1,0,0


Veamos resultados finales: 

In [23]:
df.drop(['Artista_favorito', 'Estado'], axis=1, inplace=True)
df

,Edad,Salario,Artista_favorito_Cristian_Castro,Artista_favorito_Luis_Miguel,Estado_Jalisco,Estado_Sinaloa,Estado_CDMX
0,45.781889,31576.333799,1,0,1,0,0
1,54.303289,47648.067980,1,0,1,0,0
2,48.112729,65000.000000,0,1,0,1,0
3,65.000000,34330.567827,0,1,1,0,0
4,27.137411,44627.927179,0,1,1,0,0
5,18.000000,50390.778827,0,1,1,0,0
6,24.737619,15000.000000,0,1,0,0,1
7,27.584333,23621.972569,1,0,1,0,0
8,22.215258,37153.991419,0,1,1,0,0
9,46.863818,41774.881238,0,1,1,0,0


Lo importante de ejecutar un _one hot encoding_ es **darle importancia** a una sola categoría de una única variable por columna. 

El dataset crecerá (en columnas) así: 

$$
\sum_{i=0}^{N=\text{variables categóricas}} \text{cantidad de categorías}_i
$$

En el caso de `Artista favorito`, el dataset generó dos columnas extras. Claro que una vez generadas estas podemos despreciar la variable original que ya no sirve de nada. 

#### ¿Qué pasa con la columna negación? 

El caso anterior era solo de dos columnas pero aun así puede aplicar esto. Esta estrategia está pensada para no generar tantas columnas, de hecho generará $x-1$, donde $x$ es la cantidad de categorías. 

La última columna en realidad es la negación de todas las demás, donde todas las demás son 0, ésta será 1 por eliminación. Veamos un ejemplo más concreto: 

In [33]:
df['Estado'].value_counts()

Estado
Jalisco    8
Sinaloa    1
CDMX       1
Name: count, dtype: int64

In [34]:
# Obtenemos datos únicos de la variable
datos_unicos = df['Estado'].unique()


In [ ]:
# Recorremos cada columna e identificamos los 1s
for dato_unico in datos_unicos:
    df[f'Estado_{dato_unico}'] = df['Estado'].apply(lambda x: 1 if x == dato_unico else 0)

Observamos que aquí cuando el estado no es Jalisco ni CDMX, no queda de otra más que ser Sinaloa (tercer fila). Cuando no eres Sinaloa ni CDMX, no queda de otra más que ser Jalisco (primera fila). 

Esta misma inferencia podemos dejársela al modelo y generar `k-1` _dummy variables_, es decir, eliminar la primer/última categoría. 

En este momento ya podemos tirar nuestras variables originales "Artista favorito" y "Estado".

Hagamos el experimento de modelar la variable `Salario` con solo el insumo numérico (edad) y con todas las demás. 

#### Con todas

Vamos a normalizar todas las variables. Técnicamente aquí tenemos _leakage_ por no separar los sets antes, pero por el momento oimitiremos esa validación al estar fuera del scope de la sesión. 

In [24]:
# Separacion de sets
predictores = [c for c in df.columns if c != 'Salario']

df[predictores] = (df[predictores] -df[predictores].mean()) / df[predictores].std()
df

,Edad,Salario,Artista_favorito_Cristian_Castro,Artista_favorito_Luis_Miguel,Estado_Jalisco,Estado_Sinaloa,Estado_CDMX
0,0.48975,31576.333799,1.449138,-1.449138,0.474342,-0.316228,-0.316228
1,1.02423,47648.067980,1.449138,-1.449138,0.474342,-0.316228,-0.316228
2,0.635945,65000.000000,-0.621059,0.621059,-1.897367,2.84605,-0.316228
3,1.69515,34330.567827,-0.621059,0.621059,0.474342,-0.316228,-0.316228
4,-0.67967,44627.927179,-0.621059,0.621059,0.474342,-0.316228,-0.316228
5,-1.252788,50390.778827,-0.621059,0.621059,0.474342,-0.316228,-0.316228
6,-0.83019,15000.000000,-0.621059,0.621059,-1.897367,-0.316228,2.84605
7,-0.651639,23621.972569,1.449138,-1.449138,0.474342,-0.316228,-0.316228
8,-0.988398,37153.991419,-0.621059,0.621059,0.474342,-0.316228,-0.316228
9,0.557611,41774.881238,-0.621059,0.621059,0.474342,-0.316228,-0.316228


In [25]:
# Modelo
train_x, test_x, train_y, test_y = train_test_split(
    df[predictores],
    df['Salario'],
    test_size=0.2,
    random_state=28
)

In [26]:
model = LinearRegression()

In [27]:
model.fit(train_x, train_y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [22]:
# Fit


In [28]:
# Scores
test_scores = model.predict(test_x)

In [29]:
# Error
error1 = mean_absolute_error(y_true=test_y, y_pred=test_scores)

In [30]:
error1

16791.264216016272

#### Con solo las numéricas (edad)

In [33]:
model = LinearRegression()

In [35]:
model.fit(train_x[['Edad']], train_y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [36]:
# Error
test_scores = model.predict(test_x[['Edad']])

In [37]:
error2 = mean_absolute_error(y_true=test_y, y_pred=test_scores)

In [38]:
error1, error2

(16791.264216016272, 17212.84978389737)

Obtuvimos un error menor al usar más variables que de solo usar numéricos habríamos desechado. 

## Ordinal encoding
Esta metodología es otra bastante común.

Trata básicamente en sustituir categorías con números. Si bien existirá una columna numérica al final "ordenada", el ordenamiento no tiene en realidad sentido y eso puede complicar el performance de modelos no basados en árboles (como las regresiones de cualquier tipo). 


Ventajas:
* Es fácil de aplicar y replicar.
* No genera columnas extras como one-hot encoding.

Desventajas:
* Crea una falsa sensación de ordenamiento.
* No es fácil de leer.
* Puede causar problemas productivos al llegar una categoría no vista en el entrenamiento.

In [39]:
df = pd.read_csv('actividad_9,1a.csv')

In [40]:
df

,Edad,Estado,Artista_favorito,Salario
0,45.781889,Jalisco,Cristian Castro,31576.333799
1,54.303289,Jalisco,Cristian Castro,47648.067980
2,48.112729,Sinaloa,Luis Miguel,65000.000000
3,65.000000,Jalisco,Luis Miguel,34330.567827
4,27.137411,Jalisco,Luis Miguel,44627.927179
5,18.000000,Jalisco,Luis Miguel,50390.778827
6,24.737619,CDMX,Luis Miguel,15000.000000
7,27.584333,Jalisco,Cristian Castro,23621.972569
8,22.215258,Jalisco,Luis Miguel,37153.991419
9,46.863818,Jalisco,Luis Miguel,41774.881238


In [41]:
df['Artista_favorito']

0    Cristian Castro
1    Cristian Castro
2        Luis Miguel
3        Luis Miguel
4        Luis Miguel
5        Luis Miguel
6        Luis Miguel
7    Cristian Castro
8        Luis Miguel
9        Luis Miguel
Name: Artista_favorito, dtype: str

Primer paso: Identificamos todos los valores únicos existentes.

In [42]:
valores_unicos = df['Artista_favorito'].unique().tolist()

Segundo paso: Sustituimos valores por números. 

In [43]:
# Generar diccionario a encodear
encoded_values = {original: indice for indice, original in enumerate(valores_unicos)}
encoded_values

{'Cristian Castro': 0, 'Luis Miguel': 1}

In [27]:
# Copiamos columna
df['Artista_numerico'] = df['Artista_favorito'].r

In [28]:
# Reemplazamos


In [29]:
# Comprobamos



En realidad este método es sencillo y no cambiaría mucho así sean mil categorías, sin embargo hay una particularidad con este método y es que genera una falsa sensación de ordenamiento. 

En nuestro set, "Luis Miguel" tiene el valor 1 mientras que "Cristian Castro" tiene el valor 0. En realidad aquí el 1 no es mayor que el 0 ni viceversa. Los valores numéricos lo son, pero estos valores solo están enmascarando una categoría. 